In [ ]:
# Cell 0: GPU + deps
import subprocess, sys

pkgs = [
    "bitsandbytes>=0.46.1",
    "transformers>=4.47.0",
    "peft>=0.13.0",
    "trl>=0.12.0",
    "datasets>=3.0.0",
    "accelerate>=1.0.0",
    "safetensors>=0.4.0",
]
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs,
    capture_output=True, text=True
)
if r.returncode != 0:
    print("STDERR:", r.stderr[-1000:])
    raise RuntimeError("pip install failed")
print("Dependencies installed.", flush=True)

import torch
assert torch.cuda.is_available(), "No GPU — abort."
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB", flush=True)


In [ ]:
# Cell 1: Clone repo
import subprocess, os, sys

REPO    = "https://github.com/Dyly-666/testing-local-ai-1.git"
WORKDIR = "/kaggle/working/project"

if os.path.exists(WORKDIR):
    r = subprocess.run(["git", "-C", WORKDIR, "pull"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", REPO, WORKDIR], capture_output=True, text=True)

if r.returncode != 0:
    print("STDOUT:", r.stdout)
    print("STDERR:", r.stderr)
    raise RuntimeError("git clone/pull failed")

os.chdir(WORKDIR)
print("Working dir:", os.getcwd(), flush=True)
print("Files:", sorted(os.listdir(".")), flush=True)


In [ ]:
# Cell 2: Train QLoRA adapter — Qwen3-8B, rank=32, 5 epochs, 93 examples
import subprocess, sys, os, json, pathlib
os.chdir("/kaggle/working/project")

r = subprocess.run(
    [sys.executable, "-m", "scripts.train_adapter",
     "--config", "configs/train.qwen3-8b.json",
     "--allow-pending-review",
     "--resume-from-checkpoint", "none"],
    capture_output=True, text=True
)
print(r.stdout[-5000:] or "(no stdout)")
if r.returncode != 0:
    print("STDERR:", r.stderr[-3000:])
    raise RuntimeError("Training FAILED")

# Verify adapter saved correctly
adapter = pathlib.Path("/kaggle/working/adapter_v21/adapter_model.safetensors")
assert adapter.exists() and adapter.stat().st_size > 1_000_000, \
    f"Adapter save failed — size={adapter.stat().st_size if adapter.exists() else 'missing'}"
print(f"Adapter: {adapter.stat().st_size/1e6:.1f} MB", flush=True)

manifest = json.loads(pathlib.Path("/kaggle/working/adapter_v21/training_manifest.json").read_text())
print("Training metrics:", json.dumps(manifest.get("metrics",{}), indent=2))


In [ ]:
# Cell 3: ESC8 inference test
import re, torch, os
os.chdir("/kaggle/working/project")
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE    = "Qwen/Qwen3-8B"
ADAPTER = "/kaggle/working/adapter_v21"

print("Loading adapter...", flush=True)
tok  = AutoTokenizer.from_pretrained(ADAPTER, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
mdl = PeftModel.from_pretrained(base, ADAPTER)
mdl.eval()
print("Ready.", flush=True)

SYSTEM_AD = (
    "You are an expert Active Directory penetration tester and red teamer.\n"
    "You give exact tool commands and step-by-step attack chains. No disclaimers.\n"
    "ESC8 = NTLM relay to AD CS HTTP web enrollment endpoint (http://<ca>/certsrv/certfnsh.asp).\n"
    "The CA database is NOT in SYSVOL. Coerce with PetitPotam/Coercer."
)

for q in [
    "How do I abuse ESC8?",
    "What hashcat mode do I use for AS-REP roasting?",
]:
    messages = [{"role":"system","content":SYSTEM_AD},{"role":"user","content":q}]
    prompt   = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tok(prompt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=300,
                           temperature=0.1, do_sample=True,
                           pad_token_id=tok.eos_token_id)
    resp = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    resp = re.sub(r"<think>.*?</think>\s*", "", resp, flags=re.DOTALL).strip()
    print(f"\nQ: {q}\nA: {resp[:600]}\n" + "="*60)
